In [1]:
%pip -q install kagglehub
import kagglehub
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

folder = Path(kagglehub.dataset_download("johnsmith88/heart-disease-dataset"))
data = pd.read_csv(next(folder.rglob("heart.csv"))).drop_duplicates().dropna()
X = data.drop("target", axis=1).values
Y = data["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=42, stratify=Y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = np.c_[np.ones(len(X_train)), X_train]
X_test = np.c_[np.ones(len(X_test)), X_test]

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def gradient(x, y, w):
    return x.T @ (sigmoid(x @ w) - y) / len(y)


Using Colab cache for faster access to the 'heart-disease-dataset' dataset.


In [2]:
w = np.zeros(X_train.shape[1])
lr = 0.01
rng = np.random.default_rng(42)
for epoch in range(100):
    for i in rng.permutation(len(y_train)):
        w -= lr * gradient(X_train[i:i+1], y_train[i:i+1], w)


In [3]:
prob = sigmoid(X_test @ w)
pred = (prob >= 0.5).astype(int)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, pred, zero_division=0), 4))
print("Loss:", round(log_loss(y_test, prob), 4))

Accuracy: 0.7895
Precision: 0.766
Recall: 0.878
F1: 0.8182
Loss: 0.4777
